In [1]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import json
from collections import Counter

import numpy as np
import pandas as pd
import spacy
from scipy.stats import entropy

from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

RANDOM_SEED = 42

POS_TAGS = ["ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", "NUM",
            "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X"]

nlp = spacy.load("en_core_web_trf")

In [2]:
def load_corpus(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return pd.DataFrame(records)


CORPUS_PATH = "../../results/generated_corpus.jsonl"

df = load_corpus(CORPUS_PATH)
print(df["label"].value_counts())

label
stego    272
cover    272
Name: count, dtype: int64


In [3]:
def get_pos_sequences(nlp, texts, batch_size=32):
    sequences = []
    for doc in nlp.pipe(texts, batch_size=batch_size):
        sequences.append([tok.pos_ for tok in doc if tok.pos_ != "SPACE"])
    return sequences


pos_seqs = get_pos_sequences(nlp, df["text"].tolist())

In [4]:
def _ngram_freq_vector(seq, n, vocab):
    grams = list(zip(*[seq[i:] for i in range(n)])) if len(seq) >= n else []
    counts = Counter(grams)
    total = sum(counts.values()) or 1
    return np.array([counts.get(g, 0) / total for g in vocab])


def extract_features(df, pos_seqs):
    uni_vocab = sorted(set(g for seq in pos_seqs for g in zip(seq)))
    bi_vocab = sorted(set(g for seq in pos_seqs for g in zip(seq, seq[1:])))

    rows = []
    for seq in pos_seqs:
        uni_vec = _ngram_freq_vector(seq, 1, uni_vocab)
        bi_vec = _ngram_freq_vector(seq, 2, bi_vocab)

        tag_counts = Counter(seq)
        tag_probs = np.array([tag_counts.get(t, 0) for t in POS_TAGS], dtype=float)
        tag_probs = tag_probs / (tag_probs.sum() or 1)
        pos_entropy = entropy(tag_probs + 1e-12, base=2)

        rows.append(np.concatenate([uni_vec, bi_vec, [pos_entropy, len(seq)]]))

    feature_names = [f"uni_{g}" for g in uni_vocab] + [f"bi_{g}" for g in bi_vocab] + ["pos_entropy", "n_tokens"]
    X = np.vstack(rows)
    y = (df["label"].values == "stego").astype(int)
    return X, y, feature_names


X, y, feature_names = extract_features(df, pos_seqs)
groups = pd.factorize(df["prompt"])[0]
print(f"Feature matrix: {X.shape[0]} texts x {X.shape[1]} features")

Feature matrix: 544 texts x 243 features


In [5]:
def make_pipelines():
    return {
        "Logistic regression": Pipeline([("scaler", StandardScaler()),
                                         ("clf", LogisticRegression(max_iter=2000, C=1.0))]),
        "Random forest": Pipeline([("clf", RandomForestClassifier(n_estimators=300, random_state=RANDOM_SEED))]),
        "SVM (RBF)": Pipeline([("scaler", StandardScaler()),
                               ("clf", SVC(kernel="rbf", random_state=RANDOM_SEED))]),
    }


def oof_scores(pipe, Xm, ym, gm, cv):
    if isinstance(pipe[-1], RandomForestClassifier):
        s = cross_val_predict(pipe, Xm, ym, cv=cv, groups=gm, method="predict_proba")[:, 1]
        return s, (s > 0.5).astype(int)
    s = cross_val_predict(pipe, Xm, ym, cv=cv, groups=gm, method="decision_function")
    return s, (s > 0).astype(int)


def all_metrics(yt, yp, ys):
    return {"accuracy": accuracy_score(yt, yp), "precision": precision_score(yt, yp, zero_division=0),
            "recall": recall_score(yt, yp, zero_division=0), "f1": f1_score(yt, yp, zero_division=0),
            "roc_auc": roc_auc_score(yt, ys)}


def evaluate_classifier_grouped(Xm, ym, gm, n_boot=1000):
    rng_b = np.random.default_rng(RANDOM_SEED)
    rows_by_pair = [np.where(gm == g)[0] for g in np.unique(gm)]
    table = []
    for name, pipe in make_pipelines().items():
        cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        ys, yp = oof_scores(pipe, Xm, ym, gm, cv)
        obs_m = all_metrics(ym, yp, ys)
        boot_m = []
        for _ in range(n_boot):
            idx = np.concatenate([rows_by_pair[i] for i in rng_b.integers(0, len(rows_by_pair), len(rows_by_pair))])
            boot_m.append(all_metrics(ym[idx], yp[idx], ys[idx]))
        boot_m = pd.DataFrame(boot_m)
        row = {"classifier": name}
        for m_name, v in obs_m.items():
            row[m_name] = v
            row[m_name + "_lo"] = boot_m[m_name].quantile(0.025)
            row[m_name + "_hi"] = boot_m[m_name].quantile(0.975)
        table.append(row)
    return pd.DataFrame(table).set_index("classifier")


grouped_results = evaluate_classifier_grouped(X, y, groups)
print(grouped_results[["accuracy", "accuracy_lo", "accuracy_hi", "precision", "recall", "f1",
                       "roc_auc", "roc_auc_lo", "roc_auc_hi"]].round(3).to_string())

                     accuracy  accuracy_lo  accuracy_hi  precision  recall     f1  roc_auc  roc_auc_lo  roc_auc_hi
classifier                                                                                                        
Logistic regression     0.803        0.772        0.836      0.796   0.816  0.806    0.867       0.836       0.897
Random forest           0.833        0.801        0.864      0.832   0.835  0.833    0.909       0.885       0.933
SVM (RBF)               0.805        0.774        0.838      0.819   0.783  0.801    0.895       0.869       0.919


In [6]:
def lr_grouped(Xm):
    pipe = make_pipelines()["Logistic regression"]
    s, p = oof_scores(pipe, Xm, y, groups, StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED))
    return {"accuracy": accuracy_score(y, p), "roc_auc": roc_auc_score(y, s)}


no_len_cols = [i for i, f in enumerate(feature_names) if f != "n_tokens"]

length_control = pd.DataFrame([lr_grouped(X), lr_grouped(X[:, no_len_cols])],
                              index=["All features", "Without text length"])
print(length_control.round(3).to_string())

                     accuracy  roc_auc
All features            0.803    0.867
Without text length     0.801    0.866


In [7]:
texts_norm = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()

rows_by_pair_all = [np.where(groups == g)[0] for g in np.unique(groups)]
rng_w = np.random.default_rng(RANDOM_SEED)
lexical_rows = []
for name, vec in [("Word n-grams (1-2)", TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
                  ("Character n-grams (2-5)", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True))]:
    pipe = Pipeline([("tfidf", vec), ("clf", LogisticRegression(max_iter=3000, C=1.0))])
    s = cross_val_predict(pipe, texts_norm, y, groups=groups, method="decision_function",
                          cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED))
    p = (s > 0).astype(int)
    boots = []
    for _ in range(1000):
        idx = np.concatenate([rows_by_pair_all[i] for i in rng_w.integers(0, len(rows_by_pair_all), len(rows_by_pair_all))])
        boots.append((accuracy_score(y[idx], p[idx]), roc_auc_score(y[idx], s[idx])))
    boots = np.array(boots)
    lexical_rows.append({"features": name,
                         "accuracy": accuracy_score(y, p),
                         "accuracy_lo": np.percentile(boots[:, 0], 2.5), "accuracy_hi": np.percentile(boots[:, 0], 97.5),
                         "precision": precision_score(y, p), "recall": recall_score(y, p), "f1": f1_score(y, p),
                         "roc_auc": roc_auc_score(y, s),
                         "roc_auc_lo": np.percentile(boots[:, 1], 2.5), "roc_auc_hi": np.percentile(boots[:, 1], 97.5)})
    pipe.fit(texts_norm, y)
    coef = pipe.named_steps["clf"].coef_[0]
    names = np.array(pipe.named_steps["tfidf"].get_feature_names_out())
    order = np.argsort(coef)
    print(f"[{name}] strongest stego features: {list(names[order[-12:]][::-1])}")
    print(f"[{name}] strongest cover features: {list(names[order[:12]])}")

lexical_results = pd.DataFrame(lexical_rows).set_index("features")
print(lexical_results.round(3).to_string())

[Word n-grams (1-2)] strongest stego features: ['and', 'but', 'has', 'and the', 'and is', 'but the', 'to be', 'even', 'and are', 'often', 'the', 'major']
[Word n-grams (1-2)] strongest cover features: ['this', 'an', 'with', 'as', 'when', 'such as', 'from', 'you', 'such', 'on', 'been', 'your']
[Character n-grams (2-5)] strongest stego features: [' and', ' and ', 'and ', 'and', 'nd ', 'nd', ' an', ', ', 'd ', 'but', ' but', ' but ']
[Character n-grams (2-5)] strongest cover features: ['h ', ' thi', 'his ', ' (', ' this', 'this', 'this ', 'thi', ' an ', 'th ', 'hi', 'his']
                         accuracy  accuracy_lo  accuracy_hi  precision  recall     f1  roc_auc  roc_auc_lo  roc_auc_hi
features                                                                                                              
Word n-grams (1-2)          0.860        0.833        0.888      0.871   0.846  0.858    0.943       0.924       0.959
Character n-grams (2-5)     0.825        0.796        0.855      0